### Machine Learning Pipeline

The dataset used for this example is obtained from [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing).

The second part consists of creating a machine learning model to predict if the client will subscribe (yes/no) a term deposit (variable y).

#### Import Dataset

In [1]:
# import libraries
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

In [ ]:
# read dataset
df = pd.read_csv("datasets/bank-full_train_test.csv")

#### Transform Dataset

In [117]:
# balance dataset
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Separate the classes
    df_y0 = df[df['y'] == 'no']
    df_y1 = df[df['y'] == 'yes']

    # Find the smaller class size
    min_size = len(df_y1)

    # Randomly sample from each class
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    # Concatenate back together
    df_balanced = pd.concat([df_y0_balanced, df_y1])

    # Shuffle the dataset
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

In [118]:
# map target
df.loc[df["y"] == "yes", "y"] = 1
df.loc[df["y"] == "no", "y"] = 0
df["y"] = df["y"].astype(int)

In [119]:
# prepare dataset for classification model
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = []
        self.BINARY_FEATURES = [
            "housing",
            "loan",
            "default",
        ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(self.DROP_COLUMNS, axis=1)
        df = self._map_binary_column_to_int(df)
        df = self._map_month_to_int(df)

        return df

    def _map_binary_column_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in self.BINARY_FEATURES:
            df[col] = df[col].map({"yes": 1, "no": 0})
        return df

    def _map_month_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        month_mapping = {
            "jan": 1,
            "feb": 2,
            "mar": 3,
            "apr": 4,
            "may": 5,
            "jun": 6,
            "jul": 7,
            "aug": 8,
            "sep": 9,
            "oct": 10,
            "nov": 11,
            "dec": 12,
        }
        df["month"] = df["month"].map(month_mapping)

        return df

In [120]:
# transform the dataset
df = Transformer().transform(df)

In [121]:
# encode categorical variables
ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df[ONE_HOT_ENCODE_COLUMNS])
encoded_df = encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
df = pd.concat([df, encoded_df], axis=1)

In [122]:
# save encoder - future use
joblib.dump(encoder, 'model_utils/encoder.pkl')

['model_utils/encoder.pkl']

#### Train and evaluate model

In [123]:
# Split the data into training and test sets
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [124]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

In [ ]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

In [127]:
# Predict on the test set
y_pred = lr.predict(X_test)

In [108]:
# Calculate metrics
f1_score = f1_score(y_test, y_pred)

### MLFlow Integration

In [110]:
# import libraries
import mlflow
from mlflow.models import infer_signature

In a new terminal, run a local Tracking Server with the following command:

`mlflow server --host 127.0.0.1 --port 8080`

In [112]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [114]:
# Create a new MLflow Experiment
mlflow.set_experiment("LR Experiment")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1_score)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model to present MLFlow capabilities")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="bank-model-basic",
    )

c:\Users\rober_aan06tc\Documents\EADA\test-github-setup\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'bank-model-basic' already exists. Creating a new version of this model...
2025/05/03 22:24:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to 

🏃 View run abrasive-eel-50 at: http://127.0.0.1:8080/#/experiments/117814093312462287/runs/3bf8a753662f4128892d79296a6af726
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/117814093312462287


Created version '2' of model 'bank-model-basic'.
